# S1: manually build the SimpFun RFSoC4x2 overlay

SimpFun evaluates `max(0, a*x + b)` through AXI-Lite registers. The three canonical cases return `0`, `11` and `17`.

This notebook calls the same application DAG as `python -m examples.regmap.simp_fun_board`.
Use the notebook on the development computer with Vitis/Vivado **2025.1**. Run cells
in order. Both AMD-tool switches default to `False`; enable each stage when ready.
The physical board test is a later step using the portable PYNQ host.

After an upstream update, restart the kernel and rerun every cell in order.
Saved outputs describe earlier runs; validate the updated source with a fresh run.

The notebook writes a fresh run under `examples/regmap/board_build/notebook/`.
The independent command-line build writes under `examples/regmap/board_build/`.
Existing overlays and acceptance records are not selected or reused by this notebook.

## 1. Select the notebook environment

In a terminal at the **new `board_packaging` checkout root**, register its environment:

```bash
source .venv/bin/activate
unset PYTHONPATH
# Install ipykernel first if this environment does not already provide it.
python -m ipykernel install --user --name waveflow-board-packaging \
  --display-name "Waveflow board_packaging"
```

Select **Waveflow board_packaging** in Jupyter or VS Code and restart the kernel.
If `ipykernel` is missing, install it with `python -m pip install ipykernel` in this
activated environment. The setup cell below prints the actual checkout, branch and
interpreter. Confirm that it points to the migrated checkout.

The kernel sets its own `WAVEFLOW_VITIS_PATH` and `WAVEFLOW_VIVADO_PATH` because a
Jupyter/VS Code kernel may not inherit a terminal's exports. Edit these variables
before the tool stages if the installation is elsewhere. This setup does not run
tool executables.

In [ ]:
import json
import os
from pathlib import Path
import subprocess
import sys
import tempfile

# Use the nearest checkout when Jupyter starts beside this notebook.
repo = next((path for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
             if (path / ".git").exists() and (path / "waveflow/board/spec.py").is_file()), None)
if repo is None:
    raise RuntimeError("Open this notebook from the checkout containing board packaging.")
os.environ.pop("PYTHONPATH", None)
os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "waveflow-mpl"))
tools_root = Path.home() / "Xilinx" / "2025.1"
os.environ.setdefault("WAVEFLOW_VITIS_PATH", str(tools_root / "Vitis/bin/vitis-run"))
os.environ.setdefault("WAVEFLOW_VIVADO_PATH", str(tools_root / "Vivado/bin/vivado"))
sys.path.insert(0, str(repo))
import waveflow
if not Path(waveflow.__file__).resolve().is_relative_to(repo):
    raise RuntimeError("Waveflow was imported from another checkout. Select the new kernel and restart it.")
branch = subprocess.run(["git", "branch", "--show-current"], cwd=repo,
                        check=True, capture_output=True, text=True).stdout.strip()
revision = subprocess.run(["git", "rev-parse", "HEAD"], cwd=repo,
                          check=True, capture_output=True, text=True).stdout.strip()
print("Checkout:", repo)
print("Branch:", branch)
print("Source commit:", revision)
print("Python:", sys.executable)
print("Waveflow:", waveflow.__file__)
for name in ("WAVEFLOW_VITIS_PATH", "WAVEFLOW_VIVADO_PATH"):
    print(name, "=", os.environ[name], "(exists:", Path(os.environ[name]).is_file(), ")")

## 2. Configure the application clock request

The board records its reference clocks and generation routes. Set the requested
PS PL0 output frequency, HLS timing target and allowed error in the next cell.
The default application request is 100 MHz. Configuration and software checks
do not invoke Vivado. When the hardware stage is enabled manually, synthesis
first resolves the clock from the PS reference and rejects an unachievable request.


In [ ]:
from examples.regmap.simp_fun_board import board_config, build_simp_fun_board_dag
from waveflow.build.build import BuildDag, SourceStep
from waveflow.build.board_steps import BitstreamStep, CollectOverlayStep

example_dir = repo / "examples/regmap"
build_parent = Path(os.environ.get("WAVEFLOW_NOTEBOOK_BUILD_ROOT",
                                  str(example_dir / "board_build/notebook"))).resolve()
build_parent.mkdir(parents=True, exist_ok=True)
run_dir = Path(tempfile.mkdtemp(prefix="run-", dir=build_parent)).resolve()
platforms_root = Path(os.environ.get("WAVEFLOW_NOTEBOOK_PLATFORMS_ROOT",
                                    str(example_dir / "board_build/platforms"))).resolve()
CLOCK_HZ = 100e6
HLS_CLOCK_HZ = CLOCK_HZ
CLOCK_TOLERANCE_PPM = 100.0
config = board_config(run_dir, clock_hz=CLOCK_HZ, hls_clock_hz=HLS_CLOCK_HZ,
                      tolerance_ppm=CLOCK_TOLERANCE_PPM, platform="rfsoc4x2_ps_control_100mhz",
                      platforms_root=platforms_root)
dag = build_simp_fun_board_dag()
print("Notebook build:", run_dir)
print("Platform identity:", config.platform_info.manifest_path)
print("Board / part:", config.board.name, config.part)
print("Reference clocks:", config.board.reference_clocks)
print("Requested PS PL / HLS target (Hz):", config.clock_request.requested_hz, config.clk_freq)
print("Steps:", dag.step_names())

## 3. Check the Python model and host protocol

This cell runs through `host_sim` and reports failures immediately. It performs
software checks without invoking Vitis/Vivado or programming the board. The
model-only layout is distinguished from the actual ABI obtained during IP export.

The canonical S1 cases are included alongside the default build vector.

In [ ]:
def run_checked(build_dag, through):
    def report(step, result):
        print(f"{step.name}: {'PASS' if result.success else 'FAIL'} {result.message}", flush=True)
    results = build_dag.run(config, through=through, force=True, on_step_end=report)
    failed = {name: result.traceback or result.message
              for name, result in results.items() if not result.success}
    if through not in results or failed:
        raise RuntimeError(f"Build did not complete {through}: {failed}")
    return results

local_results = run_checked(dag, "host_sim")
local_record = json.loads(local_results["host_sim"].path("host_sim_result").read_text())
assert local_record["passed"] and local_record["board_execution_verified"] is False
for case in local_record["cases"]:
    print({key: case[key] for key in ("x", "a", "b", "y", "passed")})

## 4. Manually run HLS, export, driver checks and BD validation

Set `RUN_BD_VALIDATION = True` in the next cell and execute it when ready.
It generates C++, synthesizes for RFSoC4x2, checks C/RTL results, exports IP,
validates the control ABI, generates and exercises the actual register driver,
and calls Vivado BD validation with the committed template.

The tool runners enforce version 2025.1. This stage ends before Vivado synthesis
and implementation. Keep this kernel session open to use its exact validation
and driver receipts in the next stage.

In [ ]:
RUN_BD_VALIDATION = False  # Set True, then run this cell manually.
validated = None
if RUN_BD_VALIDATION:
    missing = [name for name in ("WAVEFLOW_VITIS_PATH", "WAVEFLOW_VIVADO_PATH")
               if not Path(os.environ[name]).is_file()]
    if missing:
        raise FileNotFoundError(f"Set the installed tool paths first: {missing}")
    validated = run_checked(dag, "bd_validate")
    print("Clock resolution:", config.resolved_clock.manifest_path)
    print("Requested / achieved clock (Hz):", config.resolved_clock.requested_hz,
          config.resolved_clock.actual_hz)
    verification = json.loads(validated["synthesize"].path("hls_verification").read_text())
    bd_receipt = validated["bd_validate"].path("bd_validation")
    bd_record = json.loads(bd_receipt.read_text())
    assert verification["passed"]
    print("Fresh HLS verification:", verification)
    print("BD validation:", bd_receipt)
    print("Clock readback (Hz):", bd_record["verdict"]["physical_hz"])
    print("Address map:", bd_record["verdict"]["addresses"])
    print("Generated driver manifest:", validated["host_driver"].path("host_driver_manifest"))
else:
    print("AMD tools are disabled. Set RUN_BD_VALIDATION = True to run this stage.")

## 5. Manually implement and collect the overlay

After BD validation passes, set `RUN_IMPLEMENTATION = True` and run the next cell.
The continuation DAG uses the exact validation and driver receipts above, so it
does not repeat HLS synthesis/export. It runs Vivado synthesis/implementation,
checks timing and DRC, and collects BIT/HWH, driver.py and one manifest into a unique
overlay directory. XSA, reports and rebuild inputs remain in the build directory.

The notebook prints the precise manifest from this run. A collected overlay is
not yet a physical-board acceptance result.

In [ ]:
RUN_IMPLEMENTATION = False  # Set True after a successful BD validation.
overlay_manifest = None
implemented = None
if RUN_IMPLEMENTATION:
    if validated is None:
        raise RuntimeError("Run the BD validation cell successfully before implementation.")
    continuation = BuildDag()
    continuation.add(SourceStep(artifact="bd_validation",
                                path=validated["bd_validate"].path("bd_validation")))
    continuation.add(SourceStep(artifact="host_driver_manifest",
                                path=validated["host_driver"].path("host_driver_manifest")))
    continuation.add(BitstreamStep(name="bitstream", output_dir="implementation"))
    continuation.add(CollectOverlayStep(name="overlay", output_dir="overlays"))
    implemented = run_checked(continuation, "overlay")
    overlay_manifest = implemented["overlay"].path("overlay_manifest")
    print("Notebook overlay manifest:", overlay_manifest)
else:
    print("Implementation is disabled. Set RUN_IMPLEMENTATION = True after BD validation.")

## 6. Inspect this overlay and check its collected driver

The following cell uses only the overlay returned by this notebook run. After a
build it checks that overlay's portable driver against the current Python model.
When the tool stages are disabled it leaves the overlay unset.

In [ ]:
if overlay_manifest is None:
    print("No overlay was built by this notebook run.")
else:
    overlay_record = json.loads(overlay_manifest.read_text())
    print("Manifest:", overlay_manifest)
    print("Target:", overlay_record["target"])
    print("ABI SHA-256:", overlay_record["abi_sha256"])

    from examples.regmap.board.host.run_simp_fun import run, load_vectors
    check = run(backend="sim", overlay_manifest=overlay_manifest,
                cases=load_vectors()["cases"])
    assert check["passed"] and check["board_execution_verified"] is False
    print("Collected-driver model results:", [case["y"] for case in check["cases"]])
    print("Physical board execution verified: False (run the portable host on RFSoC4x2 next)")

## 7. Later: run fresh acceptance on RFSoC4x2

Copy the entire collected overlay directory as `overlay/`, and copy `examples/regmap/board/host/run_simp_fun.py` as `host/run_simp_fun.py`.
Use a fresh test directory on the board. The following commands run **on the board**
from that directory, using its PYNQ-enabled Python environment:

```bash
python host/run_simp_fun.py --backend pynq --overlay overlay/manifest.json \
  --all-cases --output board-acceptance-s1.json
```

Return the new acceptance JSON files after testing. The development-computer
simulation and build receipts do not replace this physical-board check. Choose
new output filenames for another run; the host refuses to overwrite records.

## 8. CMD: independently build a second overlay (with environment)

Run these commands manually in a Bash terminal at the **new checkout root**.
They use the same application DAG and current upstream CLI. This is an independent
full build in the command-line output root, separate from this notebook's run.

```bash
source .venv/bin/activate
unset PYTHONPATH
export WAVEFLOW_VITIS_PATH="$HOME/Xilinx/2025.1/Vitis/bin/vitis-run"
export WAVEFLOW_VIVADO_PATH="$HOME/Xilinx/2025.1/Vivado/bin/vivado"
export MPLCONFIGDIR=/tmp/waveflow-mpl

python -m examples.regmap.simp_fun_board \
  --clock-hz 100e6 \
  --platform rfsoc4x2_ps_control_100mhz \
  --platforms-root examples/regmap/board_build/platforms \
  --through overlay --force --timing
```

For an initial tool validation without implementation, change `--through overlay`
to `--through bd_validate`. For a software-only run use `--through host_sim`.
A later `--force` command repeats the selected stage's prerequisites.

The CMD overlay manifests are under
`examples/regmap/board_build/overlays/overlay-*/manifest.json`.
Select the manifest from the new successful CMD run for its separate board test.
Its portable cases are built into run_simp_fun.py.